<div align="center">

<a href="https://colab.research.google.com/github/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-01/NB01_first_agent.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 01: Artificial Intelligence in Engineering: Agents and First Steps in Python

**Artificial Intelligence Applications in Engineering (MUH-920 Mühendislikte Yapay Zeka Uygulamaları)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)<br>[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. Last update: October 2026.</sub>

</div>

## About this notebook

The notebook builds the heating agent step by step in plain Python. It first simulates one day with lists and loops, then downloads the hourly 2-metre air temperature for Isparta in January 2025 from the Open-Meteo archive of ERA5 reanalysis data [1, 2], and finally compares a simple reflex agent, a hysteresis agent and a proportional controller over the whole month. Three exercises check the Python and the engineering interpretation.

This notebook is the hands-on part of Week 1. It opens with Python step 1: Values, variables and decisions and continues with the sections listed in the table of contents. The concepts are explained on the [lecture page](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-01/lecture.html) and in the [PDF notes](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-01/Week01_Lecture_Notes.pdf). The [interactive lab](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-01/lab.html) holds the simulation, the self-assessment and the reflection, and the [week overview](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-01/README.md) gives the learning outcomes, the study path, the discipline challenges and the tasks.

## How to use this notebook

Run the cells in order with Shift+Enter. No earlier Python experience is assumed: Python step 1 teaches the Python of the week first, and every further piece of Python appears in a cell marked **Python move**, which explains the syntax right where it is first needed. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: Open them only after a genuine attempt. Cells that download public data need an internet connection. When a source is unreachable, the notebook falls back to a documented synthetic stand-in so that every later cell still runs.

## 0. Setup

**Python move: import.** Python keeps its tools in libraries. The line `import matplotlib.pyplot as plt` loads the plotting library under the short name `plt`. The helper `check` compares an exercise answer with the expected value and prints feedback.

The last lines of the setup cell add a hint to every `NameError`. This error appears when a cell uses a name that no cell has defined in the current session, for example because the cells above have not run.

In [ ]:
import json
import math
import random
import urllib.request

import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] {'Correct.' if ok else 'Not yet. Check the logic and run again.'}")

# Hint for a NameError: The usual cause in a notebook is a cell above that has not run in this session.
from IPython import get_ipython


def name_error_hint(shell, etype, value, tb, tb_offset=None):
    """Show a NameError as usual and add the hint."""
    shell.showtraceback((etype, value, tb), tb_offset=tb_offset)
    print("Hint: A name defined in a cell above exists only after that cell has run in this session. "
          "Runtime > Run before (Ctrl+F8) runs all cells above the current one.")


if get_ipython() is not None:
    get_ipython().set_custom_exc((NameError,), name_error_hint)

## Python step 1: Values, variables and decisions

> **Starting here?** Run the setup cell above first. The first code cell below checks this and stops with a message if the setup has not run in this session.

This step teaches the Python of the week with the week's own example. Read each explanation, run the cell below it and compare the output with the text; then change a value and predict the new result before running the cell again. A quick check and short exercises with immediate feedback close the step. The [Python path](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/PYTHON_PATH.md) shows how the fourteen steps of the course build on each other.

### Cells, values and variables

A notebook is a sequence of cells. Text cells explain, and code cells hold Python instructions that run when the cell is executed with Shift+Enter; the output appears directly below the cell [3]. The first building block of every program is the variable, a name that refers to a value. The equals sign assigns: `room_C = 18.4` stores the value 18.4 under the name `room_C`. It does not state an equation, so `count = count + 1` is a valid instruction that increases a counter. Names may contain letters, digits and underscores, cannot start with a digit and are case sensitive. Writing the unit into the name, as in `room_C` or `dt_min`, prevents many engineering mistakes, because Python itself does not track units.

In [ ]:
#@title Check that the setup cell has run
if "check" not in globals():
    raise RuntimeError("The setup cell above has not run in this session. "
                       "Run it first, then continue with this step.")

In [ ]:
setpoint_C = 21.0        # desired room temperature in degrees Celsius
room_C = 18.4            # measured room temperature
outside_C = -3.5         # outside air temperature
heater_on = False        # state of the heater
print(type(setpoint_C), type(heater_on))
print("room:", room_C, "outside:", outside_C)

Every value has a type. Numbers with a decimal point are floats, whole numbers are integers, text in quotation marks is a string, and `True` and `False` are booleans. The function `type` reports the type, and `print` shows its arguments separated by spaces. Text after a hash sign is a comment for human readers and is ignored by Python.

### Arithmetic and one step of the room model

Python uses the usual operators `+`, `-`, `*` and `/`, with `**` for powers, `//` for division rounded down and `%` for the remainder. Multiplication and division bind more strongly than addition and subtraction, and parentheses override the order. The room of this week loses heat to the outside in proportion to the temperature difference and gains heat while the heater runs. With a time constant tau, a heating rate r and a time step dt, one step of the model reads T_next = T + (T_out - T) dt / tau + r dt u, where u is 1 when the heater is on and 0 otherwise.

In [ ]:
tau_min = 240            # time constant of the room in minutes
rate_C_per_min = 0.12    # warming rate while the heater runs
dt_min = 5               # length of one time step in minutes
u = 1                    # heater command: 1 for on, 0 for off
room_next = room_C + (outside_C - room_C) * dt_min / tau_min + rate_C_per_min * dt_min * u
print(room_next)
minutes = 290
print(minutes // 60, "hours and", minutes % 60, "minutes")

The printed temperature may carry a small rounding residue in its last digits. Computers store decimal fractions in binary, so values such as 0.1 are close approximations. The differences lie far below any measurement precision, but they are the reason why floats should not be compared with `==`.

### Text and f-strings

Results need clear labels and units. An f-string, a string with the letter f before the opening quotation mark, inserts the value of any expression written in braces. A format specification after a colon controls the display: `:.2f` shows two decimals, `:+.2f` also shows the sign, and `:.0%` shows a fraction as a percentage.

In [ ]:
print(f"Next temperature: {room_next:.2f} °C")
print(f"Change in one step: {room_next - room_C:+.2f} K")
print(f"Share of the hour with heating: {0.37:.0%}")

### Comparisons and decisions

Comparisons such as `<`, `<=`, `>`, `>=`, `==` (equal) and `!=` (not equal) produce booleans, and `and`, `or` and `not` combine them. Python also accepts chained comparisons such as `low_C <= room_C <= high_C`, which read like mathematical notation. An `if` statement runs its indented block only when its condition is true; `elif` adds further conditions, and `else` covers all remaining cases. The colon at the end of each header line and the indentation of four spaces are part of the syntax: Indentation decides which lines belong to a block. The thermostat rule of this week, with a band from 20.5 to 21.5 °C, becomes the following code.

In [ ]:
low_C, high_C = 20.5, 21.5
comfortable = low_C <= room_C <= high_C
print("comfortable:", comfortable)

if room_C < low_C:
    heater_on = True
elif room_C > high_C:
    heater_on = False
else:
    pass                 # inside the band: keep the previous state
print("heater on:", heater_on)

The line `low_C, high_C = 20.5, 21.5` assigns two values at once. The statement `pass` does nothing. It marks the place where the previous state is deliberately kept, which is the hysteresis of the agent.

### Repeating a step

A `for` loop repeats its indented block once for every value of a sequence. `range(12)` produces the integers 0 to 11, so the loop below simulates one hour in twelve steps of five minutes. Each pass first applies the rule and then updates the temperature, and the conditional expression `1 if heater_on else 0` turns the boolean into the heater command u. Week 2 treats loops in more depth.

In [ ]:
room = 18.4
heater_on = False
for step in range(12):   # 12 steps of 5 minutes: one hour
    if room < low_C:
        heater_on = True
    elif room > high_C:
        heater_on = False
    u = 1 if heater_on else 0
    room = room + (outside_C - room) * dt_min / tau_min + rate_C_per_min * dt_min * u
print(f"after one hour: {room:.2f} °C, heater on: {heater_on}")

When Python cannot run a line, it stops and prints an error message whose last line names the problem. A `NameError` usually means that a variable is used before the cell that defines it has been run, and an `IndentationError` points to a block whose lines are not aligned. Reading the last line first saves a lot of time.

**Quick check.** What does print(17 // 5, 17 % 5) display?

A. 3.4 0  
B. 3 2  
C. 2 3  
D. 3.4 2

<details><summary>Show the answer</summary>

**B.** Floor division gives the whole number of times 5 fits into 17, which is 3, and the remainder is 2.

</details>

### Exercises for Python step 1

**Exercise 1.1.** With the heater off, a room at 19.0 °C and outside air at -6.0 °C, compute the room temperature after one step of 5 minutes with the model above and store it in `py_next_off`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py1_next_off_reference():
    return 19.0 + (-6.0 - 19.0) * 5 / 240

In [ ]:
py_next_off = None   # your calculation
check("Exercise 1.1", py_next_off, _py1_next_off_reference())

**Exercise 1.2.** Write an `if`, `elif` and `else` statement that stores the text "heat" in `py_mode` when `t_test` is below 20.5, "idle" when it is above 21.5 and "hold" otherwise. Run it with `t_test = 21.0` for the check, then try other temperatures.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py1_mode_reference():
    t = 21.0
    return "heat" if t < 20.5 else ("idle" if t > 21.5 else "hold")

In [ ]:
t_test = 21.0
py_mode = None       # replace with your if statement
check("Exercise 1.2", py_mode, _py1_mode_reference())

**Exercise 1.3.** A heating cycle lasts 1000 minutes. Use `//`, `%` and an f-string to store the text "16 h 40 min" in `py_duration`, computed from the number 1000 rather than typed by hand.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py1_duration_reference():
    m = 1000
    return f"{m // 60} h {m % 60} min"

In [ ]:
py_duration = None
check("Exercise 1.3", py_duration, _py1_duration_reference())

## 1. Variables and a first decision

**Python move: variables and f-strings.** A name followed by `=` stores a value. Numbers with a decimal point are floats. An f-string such as `f"{x:.1f}"` inserts a value into text with one decimal.

In [ ]:
set_point = 21.0          # desired room temperature in degrees Celsius
room = 19.4               # current measurement
heater_on = room < set_point   # a comparison gives True or False
print(f"Room {room:.1f} C, set point {set_point:.1f} C, heater on: {heater_on}")

## 2. One day of a room with lists and loops

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

The room loses heat to the outside and gains heat from the heater. A first-order model updates the temperature every five minutes:

`room = room + (outside - room) * dt / tau + gain * heater`

**Python move: lists and for loops.** `range(288)` produces the numbers 0 to 287, one for each five-minute step of a day. `append` adds an element to the end of a list. The indented lines belong to the loop.

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "set_point"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
dt, tau, heat_rate = 5.0, 240.0, 0.12     # minutes, minutes, degrees per minute at full power
outside_day = [1.5 - 5.0 * math.cos(2 * math.pi * (k * dt / 60 - 4) / 24) for k in range(288)]
room, history, heater_trace = 19.0, [], []
for k in range(288):
    heater = 1 if room < set_point else 0      # simple reflex rule
    room = room + (outside_day[k] - room) * dt / tau + heat_rate * dt * heater
    history.append(room)
    heater_trace.append(heater)
print(f"Final temperature {history[-1]:.2f} C, heater on for {sum(heater_trace) * dt / 60:.1f} hours")

**Python move: a first plot.** `plt.plot(x, y)` draws a line. Labels and a legend make the figure readable without the code.

In [ ]:
hours = [k * dt / 60 for k in range(288)]
plt.figure(figsize=(8, 3.2))
plt.plot(hours, history, label="room")
plt.plot(hours, outside_day, label="outside")
plt.axhline(set_point, color="gray", ls="--", lw=0.8, label="set point")
plt.xlabel("hour of day"); plt.ylabel("temperature (C)"); plt.legend(); plt.title("Simple reflex agent, one synthetic day")
plt.show()

## 3. Real weather: Isparta in January 2025

> **Starting here?** Run the setup cell at the top of the notebook first, or choose **Runtime > Run before** (Ctrl+F8). The first code cell below checks this and stops with a message if the setup has not run in this session.

The Open-Meteo Historical Weather API serves ERA5 reanalysis data without an API key [1, 2]. The request asks for the hourly 2-metre air temperature at 37.76 N, 30.55 E. The answer arrives as JSON text, which `json.load` turns into Python objects.

**Python move: dictionaries, first contact.** JSON objects become dictionaries, which map names to values: `data["hourly"]["temperature_2m"]` reads the list of temperatures. Week 2 treats dictionaries in detail.

**Python move: try and except.** Network access can fail. The `try` block attempts the download, and the `except` block builds a synthetic January with a daily cycle and random weather so that the notebook still runs.

In [ ]:
#@title Check that the setup cell has run
if "check" not in globals():
    raise RuntimeError("The setup cell at the top of the notebook has not run in this session. "
                       "Run it first, or choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
URL = ("https://archive-api.open-meteo.com/v1/archive?latitude=37.76&longitude=30.55"
       "&start_date=2025-01-01&end_date=2025-01-31&hourly=temperature_2m&timezone=Europe%2FIstanbul")
try:
    with urllib.request.urlopen(URL, timeout=20) as response:
        data = json.load(response)
    outside_hourly = [t for t in data["hourly"]["temperature_2m"] if t is not None]
    source = "Open-Meteo ERA5 archive, Isparta, January 2025"
except Exception as error:
    print("Download failed, using a synthetic January instead:", type(error).__name__)
    random.seed(2026)
    outside_hourly, level = [], 1.0
    for h in range(31 * 24):
        if h % 24 == 0:
            level = 0.7 * level + random.gauss(0.3, 1.6)     # day-to-day weather changes
        outside_hourly.append(level - 5.0 * math.cos(2 * math.pi * (h - 4) / 24) + random.gauss(0, 0.6))
    source = "synthetic January (fallback)"
print(source, "| hours:", len(outside_hourly), "| min", round(min(outside_hourly), 1), "| max", round(max(outside_hourly), 1))

### Your turn, exercise 1

Count the hours in which the outside temperature was below 0 C. Use a `for` loop, an `if` statement and a counter that starts at 0.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _hours_below_zero_reference():
    return sum(1 for t in outside_hourly if t < 0)

In [ ]:
hours_below_zero = None   # your computation
check("Exercise 1", hours_below_zero, _hours_below_zero_reference())

## 4. Agents as functions

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

**Python move: def.** A function packages a rule so that it can be reused. `def reflex_agent(room, heater, set_point):` defines a function with three inputs, and `return` gives back the action. The hysteresis agent needs the previous heater state, which makes it a model-based reflex agent [4].

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "outside_hourly", "source"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
def reflex_agent(room, heater, set_point, band):
    return 1 if room < set_point else 0


def hysteresis_agent(room, heater, set_point, band):
    if room < set_point - band / 2:
        return 1
    if room > set_point + band / 2:
        return 0
    return heater          # inside the dead band: keep the previous action


def proportional_agent(room, heater, set_point, band):
    u = 0.9 * (set_point - room) + 0.5
    return min(1.0, max(0.0, u))   # a heater power between 0 and 1


def simulate(agent, outside, set_point=21.0, band=1.0, tau=240.0, power_kw=3.0, steps_per_hour=12):
    """Run an agent through hourly outside temperatures and return the room trace and three scores."""
    dt = 60 / steps_per_hour
    room, heater, switches, energy, abs_error = 19.0, 0, 0, 0.0, 0.0
    trace = []
    for t_out in outside:
        for _ in range(steps_per_hour):
            new = agent(room, heater, set_point, band)
            if (new > 0.5) != (heater > 0.5):
                switches += 1
            heater = new
            room += (t_out - room) * dt / tau + 0.12 * power_kw / 3 * dt * heater
            energy += power_kw * heater * dt / 60
            abs_error += abs(room - set_point)
            trace.append(room)
    return trace, {"mean_abs_error": abs_error / len(trace), "energy_kwh": energy, "switches": switches}

In [ ]:
results = {}
for name, agent in [("simple reflex", reflex_agent), ("hysteresis", hysteresis_agent), ("proportional", proportional_agent)]:
    trace, score = simulate(agent, outside_hourly)
    results[name] = (trace, score)
    print(f"{name:14s} | mean |error| {score['mean_abs_error']:.2f} C | energy {score['energy_kwh']:.0f} kWh | switches {score['switches']}")

In [ ]:
plt.figure(figsize=(9, 3.4))
first_days = 3 * 24 * 12
for name, (trace, _) in results.items():
    plt.plot([k / 12 / 24 for k in range(first_days)], trace[:first_days], label=name, lw=1.3)
plt.axhline(21, color="gray", ls="--", lw=0.8)
plt.xlabel("day"); plt.ylabel("room temperature (C)"); plt.title(f"Three agents, first three days ({source})"); plt.legend(); plt.show()

### Your turn, exercise 2

Many relays are rated for a limited number of switching cycles. Find the smallest dead band, as a multiple of 0.1 C, for which the hysteresis agent switches fewer than 1000 times in the month. Try values by hand first, then write a loop that tests 0.0, 0.1, 0.2 and so on and stops at the first band that meets the limit. Report the mean absolute error at that band as well.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _smallest_band_reference(limit=1000):
    for i in range(0, 61):
        if simulate(hysteresis_agent, outside_hourly, band=i / 10)[1]["switches"] < limit:
            return i / 10

In [ ]:
my_band = None   # your answer, for example 1.3
if my_band is not None:
    print(simulate(hysteresis_agent, outside_hourly, band=my_band)[1])
check("Exercise 2", my_band, _smallest_band_reference())

## 5. The trade-off curve

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

A loop over dead-band values turns the choice into an engineering trade-off curve. Each point is one simulated month.

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "hysteresis_agent", "outside_hourly", "proportional_agent", "simulate"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
bands = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.5, 2.0, 2.5, 3.0]
errors, switches = [], []
for b in bands:
    _, s = simulate(hysteresis_agent, outside_hourly, band=b)
    errors.append(s["mean_abs_error"]); switches.append(s["switches"])
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(switches, errors, marker="o")
for b, x, y in zip(bands, switches, errors):
    ax.annotate(f"{b}", (x, y), textcoords="offset points", xytext=(4, 4), fontsize=8)
ax.set_xlabel("switching cycles in the month"); ax.set_ylabel("mean |error| (C)"); ax.set_title("Dead band as a design parameter")
plt.show()

### Your turn, exercise 3

The proportional agent does not switch a relay: It modulates power. Compute its energy use in kWh for the month and compare it with the hysteresis agent with a 1 C band. Store the difference, proportional minus hysteresis, in `energy_difference`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _energy_difference_reference():
    return simulate(proportional_agent, outside_hourly)[1]["energy_kwh"] - simulate(hysteresis_agent, outside_hourly, band=1.0)[1]["energy_kwh"]

In [ ]:
energy_difference = None   # your computation
check("Exercise 3", energy_difference, _energy_difference_reference(), tol=1e-3)

## Next steps

Take the [self-assessment](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-01/lab.html#check) in the interactive lab, then answer the [reflection prompts](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-01/lab.html#reflect) and export the learning log. The [discipline challenges](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-01/README.md#discipline-challenges), the [weekly task](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-01/README.md#weekly-task) and the [research assignment](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-01/README.md#research-and-report-assignment-optional) are listed in the week overview, and [Week 2](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-02/README.md) starts with its overview.

## References

[1] Hersbach, H., Bell, B., Berrisford, P., et al. (2020). The ERA5 global reanalysis. *Quarterly Journal of the Royal Meteorological Society*, *146*(730), 1999-2049. <https://doi.org/10.1002/qj.3803>

[2] Open-Meteo (2026). *Open-Meteo free weather API: Historical Weather API and Elevation API (data licensed under CC BY 4.0)*. <https://open-meteo.com>

[3] Kluyver, T., Ragan-Kelley, B., Pérez, F., et al. (2016). Jupyter Notebooks: A publishing format for reproducible computational workflows. In *Positioning and Power in Academic Publishing: Players, Agents and Agendas* (pp. 87-90). IOS Press. <https://doi.org/10.3233/978-1-61499-649-1-87>

[4] Russell, S., & Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4th ed.). Pearson.